## Imports

In [1]:
import os
from pathlib import Path
import json
import math
from keras import Model, layers
from keras.applications import EfficientNetV2S
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
from keras.utils import image_dataset_from_directory

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")

# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
# tf.config.optimizer.set_jit(True)
# print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']


## Model definitions

In [3]:
class TransferEfficientNetV2S(Model):
    """
    Pre-trained EfficientNetV2S.
    Note: EfficientNetV2 models include internal rescaling/normalisation.
    Augmentation is handled externally via tf.data.Dataset (Albumentations).
    """

    def __init__(self, num_classes, dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="transfer_effnetv2s")
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        self.base = EfficientNetV2S(
            include_top=False, 
            weights='imagenet' # Ensure weights are loaded
        )

        # Freeze the base model if you only want to train the head initially
        self.base.trainable = False 

        self.gap_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(dropout_rate)
        self.dense_layer = layers.Dense(self.num_classes, activation="softmax")

    def unfreeze_base(self, n_freeze=350):
        """
        Phase 2: unfreeze the top layers of the base for fine-tuning.
        n_freeze: number of early layers to keep frozen (they learn generic features
                  that transfer well and don't need retraining).
        """
        self.base.trainable = True
        for i, layer in enumerate(self.base.layers):
            # RULE A: Freeze the first N layers (low-level features)
            if i < n_freeze:
                layer.trainable = False
            
            # RULE B: Freeze ALL Batch Normalization layers (for Stability)
            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
        frozen = sum(1 for l in self.base.layers if not l.trainable)
        total  = len(self.base.layers)
        print(f"{self.name}: {frozen}/{total} base layers frozen, {total - frozen} unfrozen")

    def get_config(self):
        # Obtain the base config from the parent class
        config = super().get_config()
        # Add custom parameters to the config
        config.update({
            "num_classes": self.num_classes,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        # Pass inputs directly to EfficientNet (it will rescale them internally)
        x = self.base(inputs, training=training)

        x = self.gap_layer(x)
        x = self.dropout_layer(x, training=training)
        return self.dense_layer(x)

## Config and data loading

In [4]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
# 384×384: native resolution for EfficientNetV2S (significant accuracy gain over 224)
# Note: ~2.9× more pixels per image — reduce batch_size if you hit OOM on GPU
IMAGE_SIZE     = (384, 384)
BATCH_SIZE     = 16       # adjust based on your GPU's VRAM (e.g., 8 or 16 for 8GB, 32+ for 16GB)
PHASE1_EPOCHS  = 25       # frozen-base head training
PHASE2_EPOCHS  = 40       # fine-tuning (EarlyStopping will cut this short)
PHASE1_LR      = 1e-3     # higher LR — only head is updating
PHASE2_LR      = 1e-5     # ~100× lower LR — prevent destroying pretrained weights
N_CLASSES      = 23

data_dir_path = Path("..\wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

# 1. Load raw images (batched) from directories
train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
def mixup(images, labels, alpha=0.4):
    images = tf.cast(images, tf.float32)
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

# Applies mixup augmentation to the training dataset.
# We use map() to apply the mixup function to each batch of images and labels.
# The num_parallel_calls=AUTOTUNE argument allows TensorFlow to determine the optimal number of parallel calls for performance.
# Finally, we call prefetch(AUTOTUNE) to allow the dataset to fetch batches in the background while the model is training, improving performance.
train_ds_mixed = train_ds.map(mixup, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


## Weights

In [5]:
# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


## Model instantiation

In [6]:
clear_session() # Clear previous models from memory before instantiating new ones.

model = TransferEfficientNetV2S(num_classes=N_CLASSES)

## Metrics and loss

In [7]:
def make_metrics(num_classes):
    """Return a fresh set of metric instances (metrics are stateful — each model needs its own)."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]


## Learning rate schedule

In [8]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Cosine annealing with linear warmup.

    Warmup: LR ramps linearly from 0 to base_lr over the first warmup_epochs.
    This prevents the randomly initialised head from producing large gradients
    that destabilise the pretrained base at the start of training.

    Cosine decay: LR then follows a cosine curve from base_lr down to ~0.
    Finds better minima than step-decay or exponential decay in practice.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler


## Phase 1 — Train heads with frozen base

Only the GAP + Dropout + Dense head is updated.  
The pretrained base is completely frozen.


In [9]:
print(f"\n{'='*60}")
print(f"Phase 1 training: {model.name}")
print(f"{'='*60}")


model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE1_LR, weight_decay=1e-6),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase1_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase1_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE1_LR, PHASE1_EPOCHS, warmup_epochs=3)
    ),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase1_fit_data = history

print("\nPhase 1 complete.")



Phase 1 training: transfer_effnetv2s
Epoch 1/25
583/583 [==============================] - ETA: 0s - loss: 2.7995 - accuracy: 0.2573 - auc: 0.6648 - f1_score: 0.2096
Epoch 1: val_loss improved from inf to 2.18514, saving model to Checkpoints\ckpt_phase1_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


583/583 [==============================] - 236s 376ms/step - loss: 2.7995 - accuracy: 0.2573 - auc: 0.6648 - f1_score: 0.2096 - val_loss: 2.1851 - val_accuracy: 0.4995 - val_auc: 0.9084 - val_f1_score: 0.4713 - lr: 3.3333e-04
Epoch 2/25
583/583 [==============================] - ETA: 0s - loss: 2.3828 - accuracy: 0.4468 - auc: 0.7398 - f1_score: 0.3668
Epoch 2: val_loss improved from 2.18514 to 1.83981, saving model to Checkpoints\ckpt_phase1_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


583/583 [==============================] - 229s 392ms/step - loss: 2.0491 - accuracy: 0.6199 - auc: 0.7786 - f1_score: 0.5212 - val_loss: 1.5137 - val_accuracy: 0.7164 - val_auc: 0.9674 - val_f1_score: 0.6958 - lr: 1.2213e-04
Epoch 22/25
583/583 [==============================] - ETA: 0s - loss: 2.0325 - accuracy: 0.6195 - auc: 0.7790 - f1_score: 0.5254
Epoch 22: val_loss improved from 1.51369 to 1.50923, saving model to Checkpoints\ckpt_phase1_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_effnetv2s.tf\assets


583/583 [==============================] - 222s 380ms/step - loss: 2.0325 - accuracy: 0.6195 - auc: 0.7790 - f1_score: 0.5254 - val_loss: 1.5092 - val_accuracy: 0.7184 - val_auc: 0.9679 - val_f1_score: 0.6975 - lr: 7.9373e-05
Epoch 23/25
583/583 [==============================] - ETA: 0s - loss: 2.0523 - accuracy: 0.6202 - auc: 0.7772 - f1_score: 0.5234
Epoch 23: val_loss did not improve from 1.50923
583/583 [==============================] - 129s 221ms/step - loss: 2.0523 - accuracy: 0.6202 - auc: 0.7772 - f1_score: 0.5234 - val_loss: 1.5113 - val_accuracy: 0.7169 - val_auc: 0.9679 - val_f1_score: 0.6966 - lr: 4.5184e-05
Epoch 24/25
583/583 [==============================] - ETA: 0s - loss: 2.0399 - accuracy: 0.6207 - auc: 0.7794 - f1_score: 0.5232
Epoch 24: val_loss did not improve from 1.50923
583/583 [==============================] - 131s 224ms/step - loss: 2.0399 - accuracy: 0.6207 - auc: 0.7794 - f1_score: 0.5232 - val_loss: 1.5106 - val_accuracy: 0.7164 - val_auc: 0.9680 - val_

In [10]:
phase1_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase1_eval_data

{'loss': 1.513331413269043,
 'accuracy': 0.726508378982544,
 'auc': 0.9674205183982849,
 'f1_score': 0.7086555361747742}

## Phase 2 — Fine-tune unfrozen base layers

Unfreeze the top portion of the pretrained base and retrain at a much lower LR.  
Early layers learn generic features (edges, textures) that transfer well — keep them frozen.  
Later layers learn task-specific patterns — retrain these on art data.


In [11]:
print(f"\n{'='*60}")
print(f"Phase 2 fine-tuning: {model.name}")
print(f"{'='*60}")

# Unfreeze top layers — defaults are set inside each model class
model.unfreeze_base()

# Recompile at ~100× lower LR to avoid overwriting pretrained representations
model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE2_LR, weight_decay=1e-7),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase2_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase2_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE2_LR, PHASE2_EPOCHS, warmup_epochs=2)
    ),
    # More patience in Phase 2 — improvements are smaller and slower
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase2_fit_data = history

print("\nPhase 2 complete.")



Phase 2 fine-tuning: transfer_effnetv2s
transfer_effnetv2s: 382/513 base layers frozen, 131 unfrozen
Epoch 1/40
583/583 [==============================] - ETA: 0s - loss: 2.0412 - accuracy: 0.6334 - auc: 0.7869 - f1_score: 0.5286
Epoch 1: val_loss improved from inf to 1.46298, saving model to Checkpoints\ckpt_phase2_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


583/583 [==============================] - 299s 484ms/step - loss: 2.0412 - accuracy: 0.6334 - auc: 0.7869 - f1_score: 0.5286 - val_loss: 1.4630 - val_accuracy: 0.7354 - val_auc: 0.9721 - val_f1_score: 0.7162 - lr: 5.0000e-06
Epoch 2/40
583/583 [==============================] - ETA: 0s - loss: 1.9522 - accuracy: 0.6671 - auc: 0.7892 - f1_score: 0.5614
Epoch 2: val_loss improved from 1.46298 to 1.40846, saving model to Checkpoints\ckpt_phase2_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


583/583 [==============================] - 280s 480ms/step - loss: 1.5836 - accuracy: 0.8512 - auc: 0.8192 - f1_score: 0.7192 - val_loss: 1.1243 - val_accuracy: 0.8424 - val_auc: 0.9887 - val_f1_score: 0.8267 - lr: 6.8193e-08
Epoch 40/40
583/583 [==============================] - ETA: 0s - loss: 1.5516 - accuracy: 0.8670 - auc: 0.8113 - f1_score: 0.7389
Epoch 40: val_loss improved from 1.12433 to 1.12426, saving model to Checkpoints\ckpt_phase2_transfer_effnetv2s.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_effnetv2s.tf\assets


583/583 [==============================] - 281s 481ms/step - loss: 1.5516 - accuracy: 0.8670 - auc: 0.8113 - f1_score: 0.7389 - val_loss: 1.1243 - val_accuracy: 0.8424 - val_auc: 0.9888 - val_f1_score: 0.8267 - lr: 1.7078e-08

Phase 2 complete.


In [12]:
phase2_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase2_eval_data

{'loss': 1.122560977935791,
 'accuracy': 0.8511374592781067,
 'auc': 0.9873210787773132,
 'f1_score': 0.8418701887130737}